In [1]:
%load_ext juliacall

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [ ]:
%%julia

# partial pivoting is introduced in this code cell
# Solves a system of linear equations Ax = b is faster bevcause it directly back substitutes after ref(A) is found, instead of finding rref(A) and then back substituting
# determinant function now uses partial pivoting to avoid overflow and roundoff errors and it also takes in account the number of row swaps to determine the sign of the determinant


function ref(M :: Matrix{<:Number})
    U = Float64.(M)
    sz = size(U)
    i = 1
    rank = 0
    h = 1 
    while i <= sz[1] && h <= sz[2] # traversing both rows and columns
        pivot_row = i
        pivot = abs(U[i, h])
        
        # Scan ALL rows below row i in the current column h
        segment = @view U[i+1:sz[1], h]
      @inbounds @simd  for meow in 1:1:sz[1]-i
            temp = abs(segment[meow]) 
            if temp > pivot # partial pivoting is occuring here
                pivot = temp
                pivot_row = meow + i
            end
        end
        
        # If the largest element in this column is 0 then skip to the next column
        # Because a number like 0 or near zero leads to large numbers ( when divided by 0 or near zero)
        # which leads to overflow and roundoff errors
        if pivot == 0
            h += 1
            continue
        end
        
        # pivot found hence increase the rank by 1
        rank += 1
        h_carry = h
        
        # rows of U swapped
        temp = U[i, :]
        U[i, :] = U[pivot_row, :]
        U[pivot_row, :] = temp

        
        pivot_val = U[i, h_carry]
        @inbounds @simd for j in i+1:1:sz[1]
            mbp = U[j, h_carry] / pivot_val 
            @inbounds @simd for k = 1:1:sz[2]
                U[j, k] = U[j, k] - mbp * U[i, k]
            end
        end
        
        i += 1
        h += 1
    end
    return U, rank,sz[2]-rank
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D,rank,nullity = ref(U)
    
    sz = size(D)
 @inbounds  for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
   @inbounds   for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
      @inbounds @simd      for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
      @inbounds @simd for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
               @inbounds @simd     for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end




#--------------------------------------------------------------------------------------------------------------------------------------------

function solveLSE(D::Matrix{<:Number})
    M ,rank,nullity = ref(D)
    sz = size(M)
    for i in 1:sz[1]
        count = 0 
        for j in 1:(sz[2]-1)
            if M[i, j] != 0
                count += 1
                break # Pivot is found, hence this row will give a valid solution hence break
            end
        end
        
        # Check if in A|b the last column is non-zero while all other columns are zero, which indicates no solution exists
        if count == 0 && M[i, sz[2]] != 0
            return ("No solution exists",) # Wrap in a tuple to match your Python unpacker expectations
        end
    end
    if rank < (sz[2]-1)
        particular_vector ,linear_combination_of_vectors = solutions(M,rank)
        return "infinite soln" , particular_vector ,linear_combination_of_vectors
    else
        return "Unique solution",solutions(M,rank) 
    end
end
#-------------------------------------------------------------------------------------------------------------------------------------------

function solutions(D::Matrix{<:Number},r::Number) # D = ref(A) : D is augmented matrix
    U = D
    rank = r
    sz = size(U)    

    particular_vector = zeros(Float64,sz[2]-1,1)
    row_status = zeros(Int64,sz[1],2) 
    # rows of row_status = number of rows
    # first column tells wheather a row is pivot row(1) or free row(0) 
    # second column tells in which column that pivot is 
    column_status = zeros(Int64,sz[2]-1,1)
    # columns of column_status = number of columns in A which is A|b = U
    # first row tells wheather a column is pivot column(1) or free column(0)
    
 @inbounds  for i in 1:1:sz[1]
    @inbounds     for j in 1:1:sz[2]-1
            if U[i,j] !=0
                row_status[i,1] = 1
                row_status[i,2] = j
                column_status[j] =1
                break
            
            end
        end
    end

@inbounds for i in sz[1]:-1:1
        if row_status[i,1] ==0
             nothing
        else
            pivot_column = row_status[i,2]
            b_comp = U[i,sz[2]]
          @inbounds @simd  for doraemon in pivot_column+1:1:sz[2]-1
                if column_status[doraemon] == 0
                    nothing
                else
                    b_comp -= U[i,doraemon]*particular_vector[doraemon]
                end
            end
            particular_vector[pivot_column]= b_comp/U[i,pivot_column]
        end
    end 

       linear_combination_of_vectors = Matrix{Float64}[]   
@inbounds for j in 1:1:sz[2]-1
            if column_status[j] == 0 #= We only care about free columns where column_status[j] == 0
                                         Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                        =#
            
            basis_vector = zeros(Float64,sz[2]-1,1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
          @inbounds for i in sz[1]:-1:1
                if row_status[i,1] ==0
                    nothing
                else
                    pivot_column = row_status[i,2]
                    b_comp = 0
                    @inbounds @simd  for doraemon in pivot_column+1:1:sz[2]-1
                        b_comp -= U[i,doraemon]*basis_vector[doraemon]
                                     end
                    basis_vector[pivot_column]= b_comp/U[i,pivot_column]
                end
            end
            push!(linear_combination_of_vectors,basis_vector)
        end
    end
return particular_vector, linear_combination_of_vectors   
end
#------------------------------------------------------------------------------------------------------------------------------------------------
function inverse(M::Matrix{<:Number})
    sz = size(M)
     U = Float64.(M)
    if sz[1] != sz[2]
        return throw(ArgumentError("Non Square Matrices dont have inverse"))
    #= elseif ref(M)[2] <sz[2]
        return "Inverse do NOT exist" =#
    elseif ref(M)[2] < sz[2]
        return "Inverse Do Not Exist"
    else
       I_mat =  zeros(Float64,sz[1],sz[2])
        for i in 1:1:sz[1]
            I_mat[i,i] = 1
        end
          i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp1 = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp1

                                temp2 = I_mat[meow, :]
                                I_mat[meow, :] = I_mat[i, :]
                                I_mat[i, :] = temp2
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end   
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                     I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                end
            end
            i+=1
    end
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if U[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = U[i,h_carry] 
            for meow in 1:1:sz[2]
                U[i,meow] = U[i,meow]/pivot
                I_mat[i,meow] = I_mat[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = U[j,h_carry]
                    for k in 1:1:sz[2]
                        U[j,k] = U[j,k]-mbp*U[i,k]
                        I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                    end
            end 
        end
    end
    return I_mat
              
end  
end
#----------------------------------------------------------------------------------------------------------------------------------------------------
function det(M::Matrix{<:Number})
    U = Float64.(M)
    sz = size(U)
    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have determinant"))
    end
    i = 1
    swaps = 0
    h = 1
    rank = 0
    while i <= sz[1] && h <= sz[2] # traversing both rows and columns
        pivot_row = i
        pivot = abs(U[i, h])
        
        # Scan ALL rows below row i in the current column h
        segment = @view U[i+1:sz[1], h]
      @inbounds @simd  for meow in 1:1:sz[1]-i
            temp = abs(segment[meow]) 
            if temp > pivot # partial pivoting is occuring here
                pivot = temp
                pivot_row = meow + i
            end
        end
        
        # If the largest element in this column is 0 then skip to the next column
        # Because a number like 0 or near zero leads to large numbers ( when divided by 0 or near zero)
        # which leads to overflow and roundoff errors
        if pivot == 0
            h += 1
            continue
        end
        
        # pivot found hence increase the rank by 1
        rank += 1
        h_carry = h
        
        # rows of U swapped
        if pivot_row != i
            temp = U[i, :]
            U[i, :] = U[pivot_row, :]
            U[pivot_row, :] = temp
            swaps += 1
        end

        
        pivot_val = U[i, h_carry]
        @inbounds @simd for j in i+1:1:sz[1]
            mbp = U[j, h_carry] / pivot_val 
            @inbounds @simd for k = 1:1:sz[2]
                U[j, k] = U[j, k] - mbp * U[i, k]
            end
        end
        
        i += 1
        h += 1
    end
    
    if  rank<sz[2]
        return 0
    else
        determinant = 1
        for i in 1:sz[1]
            determinant *= U[i, i]
        end
    end
    return determinant*(-1)^swaps
end

det (generic function with 1 method)

In [1]:
%%julia
# this is complete RDIS file, but it still lacks partial pivoting 
# the solving processs can be made more efficint by doing direct back subjstitution after ref(A)
# the diterminant function still does not takes in account the row flips

function ref(M :: Matrix{<:Number})
    U = Float64.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                     meow = i+1
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U,rank,sz[2]-rank
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D,rank,nullity = ref(U)
    
    sz = size(D)
 @inbounds  for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
   @inbounds   for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
      @inbounds @simd      for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
      @inbounds @simd for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
               @inbounds @simd     for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end




#--------------------------------------------------------------------------------------------------------------------------------------------

function solveLSE(D::Matrix{<:Number})
    M ,rank,nullity = rref(D)
    sz = size(M)
    for i in 1:sz[1]
        count = 0 
        for j in 1:(sz[2]-1)
            if M[i, j] != 0
                count += 1
                break # Pivot is found, hence this row will give a valid solution hence break
            end
        end
        
        # Check if in A|b the last column is non-zero while all other columns are zero, which indicates no solution exists
        if count == 0 && M[i, sz[2]] != 0
            return ("No solution exists",) # Wrap in a tuple to match your Python unpacker expectations
        end
    end
    if rank < (sz[2]-1)
        particular_vector ,linear_combination_of_vectors = solutions(M,rank)
        return "infinite soln" , particular_vector ,linear_combination_of_vectors
    else
        return "Unique solution",solutions(M,rank) 
    end
end
#-------------------------------------------------------------------------------------------------------------------------------------------

function solutions(D::Matrix{<:Number},r::Number) # D = rref(A)
    M_raw = D
    rank = r
    sz = size(M_raw)    
    M = Float64.(M_raw[:,1:sz[2]-1])   
    
    column_status = zeros(Int64, sz[2]-1, 2) # pivot column = 1 , free column = 0
    # first Column  = keeps the check on pivot columns
    # second column = keeps the track in which that pivot resides
    
  @inbounds for i in 1:1:sz[1]
  @inbounds for j in 1:1:sz[2]-1
            if M[i,j] != 0
                column_status[j,1] = 1
                column_status[j,2] = i
                break
            end
        end
    end #= By the end of this code block, our column_status Matrix(or array) should have the information of pivots in which column(in binary)
    and in which row =#

    
    particular_vector = zeros(Float64,sz[2]-1,1) 
 @inbounds @simd   for p_col in 1:1:sz[2]-1 # traverse the columns
        if column_status[p_col, 1] == 1     # if a pivot is found
            row = column_status[p_col, 2]   # then look for row in which that resides 
            particular_vector[p_col] = M_raw[row, sz[2]]  # then the particular vector will be the last element of that row

            # all free variables are set to be 0
        end
    end # particular vector is done after this code block

    
    linear_combination_of_vectors = Matrix{Float64}[]   
@inbounds @simd    for j in 1:1:sz[2]-1
            if column_status[j,1] == 0 #= We only care about free columns where column 1 is 0 (i.e column_status[j,1] = 0)
                                         Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                        =#
            
            basis_vector = zeros(Float64,sz[2]-1,1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
            
            # Now, look at all the pivot columns to fill in the rest of this vector
        @inbounds @simd    for p_col in 1:1:sz[2]-1 # this loop traverses the columns 
                if column_status[p_col,1] == 1 
                    row = column_status[p_col, 2] #  note down the rows in which the pivot resides
                    basis_vector[p_col] = -M[row, j] #=  assign the element from the free column(after flipping the sign) 
                                                       to the pivot column element of the basis vector =#
                end
            end
            
            push!(linear_combination_of_vectors, basis_vector)
        end
    end
    if rank == sz[2]-1
         return particular_vector
    else
         return (particular_vector ,linear_combination_of_vectors)
    end
end
#------------------------------------------------------------------------------------------------------------------------------------------------
function inverse(M::Matrix{<:Number})
    sz = size(M)
     U = Float64.(M)
    if sz[1] != sz[2]
        return throw(ArgumentError("Non Square Matrices dont have inverse"))
    #= elseif ref(M)[2] <sz[2]
        return "Inverse do NOT exist" =#
    elseif ref(M)[2] < sz[2]
        return "Inverse Do Not Exist"
    else
       I_mat =  zeros(Float64,sz[1],sz[2])
        for i in 1:1:sz[1]
            I_mat[i,i] = 1
        end
          i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp1 = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp1

                                temp2 = I_mat[meow, :]
                                I_mat[meow, :] = I_mat[i, :]
                                I_mat[i, :] = temp2
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end   
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                     I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                end
            end
            i+=1
    end
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if U[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = U[i,h_carry] 
            for meow in 1:1:sz[2]
                U[i,meow] = U[i,meow]/pivot
                I_mat[i,meow] = I_mat[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = U[j,h_carry]
                    for k in 1:1:sz[2]
                        U[j,k] = U[j,k]-mbp*U[i,k]
                        I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                    end
            end 
        end
    end
    return I_mat
              
end  
end
#----------------------------------------------------------------------------------------------------------------------------------------------------
function det(M_raw ::Matrix{<:Number},)
    M,rank, nullity = ref(M_raw)
    sz = size(M)

    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have determinant"))
    elseif  rank<sz[2]
        return 0
    else
        determinant = 1
        for i in 1:sz[1]
            determinant *= M[i, i]
        end
    end
    return determinant
end

UsageError: Cell magic `%%julia` not found.


In [4]:
%%julia
# changed to float64 to get speed, although exact precision is lost but the accuracy is still very good
#=
solutions function is added
it gives all the solutions to the solveLSE function
which then gives a tuple
first element of tuple is the status of the system of equations
2nd element are the solutions
=#

function ref(M :: Matrix{<:Number})
    U = Float64.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                     meow = i+1
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U,rank,sz[2]-rank
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D_raw,rank,nullity = ref(U)
    D = Float64.(D_raw)
    sz = size(D)
 @inbounds  for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
   @inbounds   for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
      @inbounds @simd      for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
      @inbounds @simd for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
               @inbounds @simd     for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end




#--------------------------------------------------------------------------------------------------------------------------------------------

function solveLSE(D::Matrix{<:Number})
    M,rank,nullity = rref(D)
    sz = size(M)
    lr = @view M[sz[1],1:sz[2]-1]
    count = 0
  @inbounds for i in lr 
        if i != 0
            count +=1
            break
        else
            continue
        end
    end
    if count ==0 && M[sz[1],sz[2]] != 0
        return ("NO Solution",)
    elseif rank < (sz[2]-1)
         particular_vector ,linear_combination_of_vectors= solutions(M,rank)
         return "Infinite Solutions", particular_vector , "+ linear combination of vectors",linear_combination_of_vectors
    else
         particular_vector ,linear_combination_of_vectors= solutions(M,rank)
         return "Unique Solution", particular_vector 
    end
end
#-------------------------------------------------------------------------------------------------------------------------------------------

function solutions(D::Matrix{<:Number},r::Number) # D = rref(A)
    M_raw = D
    rank = r
    sz = size(M_raw)    
    M = Float64.(M_raw[:,1:sz[2]-1])   
    
    column_status = zeros(Int64, sz[2]-1, 2) # pivot column = 1 , free column = 0
    # first Column  = keeps the check on pivot columns
    # second column = keeps the track in which that pivot resides
    
  @inbounds for i in 1:1:sz[1]
  @inbounds for j in 1:1:sz[2]-1
            if M[i,j] != 0
                column_status[j,1] = 1
                column_status[j,2] = i
                break
            end
        end
    end #= By the end of this code block, our column_status Matrix(or array) should have the information of pivots in which column(in binary)
    and in which row =#

    
    particular_vector = zeros(Float64, sz[2]-1) 
 @inbounds @simd   for p_col in 1:1:sz[2]-1 # traverse the columns
        if column_status[p_col, 1] == 1     # if a pivot is found
            row = column_status[p_col, 2]   # then look for row in which that resides 
            particular_vector[p_col] = M_raw[row, sz[2]]  # then the particular vector will be the last element of that row

            # all free variables are set to be 0
        end
    end # particular vector is done after this code block

    
    linear_combination_of_vectors = Vector{Float64}[]   
@inbounds @simd    for j in 1:1:sz[2]-1
            if column_status[j,1] == 0 #= We only care about free columns where column 1 is 0 (i.e column_status[j,1] = 0)
                                         Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                        =#
            
            basis_vector = zeros(Float64, sz[2]-1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
            
            # Now, look at all the pivot columns to fill in the rest of this vector
        @inbounds @simd    for p_col in 1:1:sz[2]-1 # this loop traverses the columns 
                if column_status[p_col,1] == 1 
                    row = column_status[p_col, 2] #  note down the rows in which the pivot resides
                    basis_vector[p_col] = -M[row, j] #=  assign the element from the free column(after flipping the sign) 
                                                       to the pivot column element of the basis vector =#
                end
            end
            
            push!(linear_combination_of_vectors, basis_vector)
        end
    end
    if rank == sz[2]-1
         return particular_vector
    else
         return (particular_vector ,linear_combination_of_vectors)
    end
end
#------------------------------------------------------------------------------------------------------------------------------------------------
function inverse(M::Matrix{<:Number})
    sz = size(M)
     U = Float64.(M)
    if sz[1] != sz[2]
        return throw(ArgumentError("Non Square Matrices dont have inverse"))
    #= elseif ref(M)[2] <sz[2]
        return "Inverse do NOT exist" =#
    elseif ref(M)[2] < sz[2]
        return "Inverse Do Not Exist"
    else
       I_mat =  zeros(Float64,sz[1],sz[2])
        for i in 1:1:sz[1]
            I_mat[i,i] = 1
        end
          i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp1 = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp1

                                temp2 = I_mat[meow, :]
                                I_mat[meow, :] = I_mat[i, :]
                                I_mat[i, :] = temp2
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end   
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                     I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                end
            end
            i+=1
    end
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if U[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = U[i,h_carry] 
            for meow in 1:1:sz[2]
                U[i,meow] = U[i,meow]/pivot
                I_mat[i,meow] = I_mat[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = U[j,h_carry]
                    for k in 1:1:sz[2]
                        U[j,k] = U[j,k]-mbp*U[i,k]
                        I_mat[j,k] = I_mat[j,k]-mbp*I_mat[i,k]
                    end
            end 
        end
    end
    return Matrix(transpose(I_mat))
              
end  
end
#----------------------------------------------------------------------------------------------------------------------------------------------------
function det(M_raw ::Matrix{<:Number},)
    M,rank, nullity = ref(M_raw)
    sz = size(M)

    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have determinant"))
    elseif  rank<sz[2]
        return 0
    else
        determinant = 1
        for i in 1:sz[1]
            determinant *= M[i, i]
        end
    end
    return determinant
end


# ==============================================================================
# 🧪 PHASE 3 NUMERICAL TESTS
# [Note: Test validation cases auto-generated via LLM for codebase verification]
# ==============================================================================

println("--- Test Case 1: Unique Solution Linear System ---")
# System: 2x + y = 5, x + 3y = 5 
# Augmented layout matrix 
sys_unique = [2.0 1.0 5.0; 
              1.0 3.0 5.0] 
res_unique = solveLSE(sys_unique)
println("System State: ", res_unique[1])
if length(res_unique) > 1
    println("Solution Structure:\n", res_unique[2])
end
println()


println("--- Test Case 2: Infinite Solutions Matrix ---")
# System with a free variable vector requirement
sys_infinite = [1.0 2.0 3.0; 
                2.0 4.0 6.0]
res_infinite = solveLSE(sys_infinite)
println("System State: ", res_infinite[1])
if length(res_infinite) > 1
    println("Particular + Basis Arrays Layout:\n", res_infinite[2])
end
println()


println("--- Test Case 3: Matrix Determinant & Inversion Execution ---")
# Standard non-singular square configuration layout
A = [4.0 7.0; 
     2.0 6.0]
println("Determinant det(A): ", det(A))
println("Computed Inverse Matrix inverse(A):\n", inverse(A))
println()


println("--- Test Case 4: Non-Invertible Edge Case ---")
# Singular system evaluation checking safety fallback states
B = [1.0 2.0; 
     2.0 4.0]
println("Singular Determinant det(B): ", det(B))
println("Inverse Handling Response: ", inverse(B))
println("\n✅ Verification suite successfully processed.")






--- Test Case 1: Unique Solution Linear System ---
System State: Unique Solution
Solution Structure:
2.0

--- Test Case 2: Infinite Solutions Matrix ---
System State: Infinite Solutions
Particular + Basis Arrays Layout:
[3.0, 0.0]

--- Test Case 3: Matrix Determinant & Inversion Execution ---
Determinant det(A): 10.0
Computed Inverse Matrix inverse(A):
[0.6000000000000001 -0.2; -0.7000000000000001 0.4]

--- Test Case 4: Non-Invertible Edge Case ---
Singular Determinant det(B): 0
Inverse Handling Response: Inverse Do Not Exist

✅ Verification suite successfully processed.


In [ ]:
%%julia

#= 
SolveLSE function is added
it currently gives output as 
1. No solutuon
2. Infinite solutions
3. Unique Solution and the solutions also
=#

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return (U,rank,sz[2]-rank)
end
#------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})
     D_raw,rank,nullity = ref(U)
    D = Rational{Int128}.(D_raw)
    sz = size(D)
    for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
        for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
            for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
            end
            for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
                    for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
            end 
        end
    end
    return D,rank,nullity
end




#--------------------------------------------------------------------------------------------------------------------------------------------

function solveLSE(D::Matrix{<:Number})
    M,rank,nullity = rref(D)
    sz = size(M)
    lr = @view M[sz[1],1:sz[2]-1]
    count = 0
    for i in lr 
        if i != 0
            count +=1
            break
        else
            continue
        end
    end
    if count ==0 && M[sz[1],sz[2]] != 0
        return "NO Solution"
    elseif rank < (sz[2]-1)
        return "infinite solns"
    else
        
        return "Unique solution",M[:,sz[2]]
    end
end

M = [1 1 1;22 33 44]
println(solveLSE(M))

("Unique solution", Rational{Int128}[-1, 2])


In [ ]:
%%julia
# gives reduced row echelon form, and if Matrix is full rank then it gives the solution too

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return (U,rank,sz[2]-rank)
end
#---------------------------------------------------------------------------------------------------------------------------------------------

function rref(U::Matrix{<:Number})  
    D_raw,rank,nullity = ref(U)
    D = Rational{Int128}.(D_raw)
    sz = size(D)
 @inbounds for i = sz[1]:-1:1 # goes from last row to the first row
        pivot = 0
        h_carry = 0
       @inbounds for h in 1:1:sz[2] # traverses the selected row
            if D[i,h] != 0 # if a pivot is found then 
                h_carry = h
                break
            end
        end
        if h_carry != 0 # if h_carry is there then a pivot is found, hence only then we will proceed to do backward( OR upward) elimination 
            pivot = D[i,h_carry] 
 @inbounds @simd for meow in 1:1:sz[2]
                D[i,meow] = D[i,meow]/pivot
                 end
@inbounds @simd for j in i-1:-1:1 # selects/ locks the rows above the pivot row( selected by i pointer) 
                    mbp = D[j,h_carry]
    @inbounds @simd for k in 1:1:sz[2]
                        D[j,k] = D[j,k]-mbp*D[i,k]
                    end
                end 
        end
    end
    return D,rank,nullity
end
A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]
B = [1 2 3;4 5 6;7 8 9]
M = [1 1 7;5 12 7]

rref(B)

(Julia:
 3×3 Matrix{Rational{Int128}}:
  1  0  -1
  0  1   2
  0  0   0,
 2,
 1)

In [ ]:
%%julia
#= Best one so far 
speed : Float64>Int128,BigInt
GC impact : Int128>BigInt>Float64
Accuracy : BigInt>Int128> Float64
=#

# I have changed it a bit it provides : ref(A),rank, Null space dimensions

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop onlu
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return (U,rank,sz[2]-rank)
end

A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]
B = [1 2 3;4 5 6;7 8 9]
M = [1 1 7;5 12 7]

ref(A)[1]



2×3 Matrix{Rational{Int128}}:
 1  1    7
 0  7  -28

In [ ]:
%%julia
#= Best one so far 
speed : Float64>Int128,BigInt
GC impact : Int128>BigInt>Float64
Accuracy : BigInt>Int128> Float64
=#

function ref(M :: Matrix{<:Number})
    U = Rational{Int128}.(M)
    sz =size(U)
    i =1
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop only
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    break
                else
                    meow = i+1
                    
                    if meow <= sz[1]
            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return U
end

A_m = [3 6 5;0 1 7;7 8 9;1 7 2]
A = [2 1 1;4 3 3;8 7 9]
ref(A)

3×3 Matrix{Rational{Int128}}:
 2  1  1
 0  1  1
 0  0  2